# Анализ объявлений о продаже недвижимости

Цель проекта — подготовить данные и построить модель, которая оценивает стоимость жилого объекта. В этом ноутбуке я проверяю качество исходных данных и рассматриваю несколько основных зависимостей. Я сознательно ограничился графиками, которые помогают принять решения для модели.

**Задачи:** очистить данные, исследовать признаки, статистически проверить выводы, построить baseline, сравнить линейную и нелинейную модели и проанализировать ошибки.

**Этапы работы:** загрузка → очистка → EDA → статистические тесты → обучение моделей → оценка качества и ограничений.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import MODEL_FEATURES, load_raw_data, prepare_dataset

raw = load_raw_data(PROJECT_ROOT / 'data.csv.zip')
data, audit = prepare_dataset(raw)
audit

## Подготовка данных

В исходном файле было 377 185 строк. Я удалил 242 полных дубля, 398 объявлений с ежемесячной арендной ставкой, 31 403 земельных участка и 3 684 строки без подходящей цены. После очистки осталось 339 000 объектов.

Земельные участки исключены, поскольку для них неприменимы многие характеристики дома. Поле `Price/sqft` из `homeFacts` не используется: оно уже рассчитано с помощью цены и привело бы к утечке целевой переменной.

In [ ]:
data[['target', 'sqft', 'beds', 'baths', 'year_built']].describe().round(1)

## Распределение стоимости

![Распределение цены](../reports/price_distribution.png)

Медианная цена равна 339 900 долларов, а средняя — 590 541 доллар. Среднее заметно выше медианы из-за небольшого числа дорогих объектов. Поэтому при моделировании целевая переменная логарифмируется, а основной метрикой выбрана MAE, менее чувствительная к отдельным дорогим домам, чем RMSE.

In [ ]:
data['target'].quantile([0.01, 0.5, 0.9, 0.99])

## Пропущенные значения

![Пропуски](../reports/missing_values.png)

Больше всего пропусков в годе ремонта и этажности. Удалять все такие строки нельзя: выборка сильно уменьшится. Поэтому числовые пропуски в pipeline заполняются медианой, а категориальные — наиболее частым значением. Для числовых столбцов также добавляются индикаторы пропусков.

## Площадь и стоимость

![Цена и площадь](../reports/price_vs_sqft.png)

Коэффициент Спирмена между площадью и стоимостью равен 0,549. Связь положительная, но разброс остаётся большим: дома одинаковой площади могут сильно отличаться по цене из-за штата, города, возраста и других характеристик.

## Различия между штатами

![Цена по штатам](../reports/median_price_by_state.png)

Местоположение оказалось важным признаком. Среди штатов с наибольшим числом объявлений медианная стоимость меняется примерно от 150 тысяч долларов в Огайо до 799 тысяч в Нью-Йорке. Поэтому штат, город и почтовый индекс были сохранены для модели как категориальные признаки.

In [ ]:
top_states = data['state'].value_counts().head(10).index
data[data['state'].isin(top_states)].groupby('state')['target'].median().sort_values()

## Статистическая проверка выводов

Для площади проверена нулевая гипотеза об отсутствии монотонной связи с ценой. Корреляция Спирмена равна 0,549 при `p < 0,001` (313 622 наблюдения), поэтому нулевая гипотеза отвергается. Площадь сохранена среди признаков модели.

Для десяти наиболее представленных штатов применён критерий Краскела — Уоллиса. Он также показал статистически значимые различия цен (`H = 47 067`, `p < 0,001`). Размер эффекта $\varepsilon^2 = 0,158$, то есть различия имеют не только статистический, но и практический смысл. Поэтому штат, город и почтовый индекс используются при обучении.

## Результаты моделей

| Модель | MAE, $ | RMSE, $ | R² |
|---|---:|---:|---:|
| Медианный baseline | 383 788 | 909 568 | -0,079 |
| Ridge | 212 161 | 590 570 | 0,545 |
| HistGradientBoosting | 189 397 | 539 233 | 0,621 |

Нелинейная модель показала лучший результат по всем трём метрикам. Это согласуется с EDA: цена зависит от площади и местоположения нелинейно. При этом я не выполнял большой подбор гиперпараметров, поскольку задача проекта — показать полный цикл работы с данными, а не получить максимально возможную точность.

## Анализ ошибок

![Факт и прогноз](../reports/actual_vs_predicted.png)

Медианная абсолютная ошибка итоговой модели равна 62 020 долларов. Для объектов дешевле 200 тысяч MAE составляет около 48 тысяч, а для объектов дороже 1 млн — около 916 тысяч долларов. Модель хуже оценивает дорогие дома: таких объектов меньше, а их стоимость сильнее зависит от особенностей, которых нет в датасете. Поэтому сервис следует использовать как ориентировочную оценку, а не как замену профессиональной оценке недвижимости.

## Выводы

1. Цены имеют правостороннее распределение, поэтому для линейной модели полезно логарифмирование target.
2. Площадь связана с ценой, но одной площади недостаточно для точного прогноза.
3. Географические признаки необходимо сохранить из-за больших региональных различий.
4. Из-за большого числа пропусков разумнее применять заполнение внутри pipeline, а не удалять строки.
5. HistGradientBoosting выбран итоговой моделью: его MAE примерно на 50% ниже медианного baseline.
6. Наиболее дорогие объекты прогнозируются хуже, поэтому итоговая оценка имеет рекомендательный характер.